# Lab 5 - Wumpus World with Q-learning

The agent starts in the bottom-left corner. It must take the gold and come back to the start.
If it walks into a pit, or into the Wumpus while it is alive, it dies.
The agent has one arrow: it can shoot in one direction, and the arrow kills the Wumpus if it is in that line.
The agent does not know the map: it learns by playing many games (Q-learning).

In [1]:
import random

random.seed(42)  # same results every time

## The world

In this map the Wumpus stands in front of the gold, so the agent must learn to kill it first.

In [2]:
SIZE = 4
START = (3, 0)                     # bottom-left corner
WUMPUS = (1, 3)
PITS = [(0, 2), (1, 1), (3, 2)]
GOLD = (0, 3)

MOVES = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
ACTIONS = ["up", "down", "left", "right",
           "shoot up", "shoot down", "shoot left", "shoot right"]


def print_world():
    for r in range(SIZE):
        row = ""
        for c in range(SIZE):
            if (r, c) == WUMPUS:
                row += "W "
            elif (r, c) in PITS:
                row += "P "
            elif (r, c) == GOLD:
                row += "G "
            elif (r, c) == START:
                row += "S "
            else:
                row += ". "
        print(row)


print_world()

. . P G 
. P . W 
. . . . 
S . P . 


## One step in the world

The state is `(row, col, has_gold, has_arrow, wumpus_alive)`.

Rewards:
* every step: -1
* shooting the arrow: -10
* pit, or the Wumpus while it is alive: -1000 (game over)
* back to the start with the gold: +1000 (game over)

In [3]:
def shoot(r, c, direction):
    # the arrow flies in a straight line until the wall
    dr, dc = MOVES[direction]
    r, c = r + dr, c + dc
    while 0 <= r < SIZE and 0 <= c < SIZE:
        if (r, c) == WUMPUS:
            return True        # Wumpus killed
        r, c = r + dr, c + dc
    return False               # missed


def step(state, action):
    r, c, has_gold, has_arrow, wumpus_alive = state

    if action.startswith("shoot"):
        direction = action.split()[1]
        if has_arrow and wumpus_alive and shoot(r, c, direction):
            wumpus_alive = False
        has_arrow = False
        return (r, c, has_gold, has_arrow, wumpus_alive), -10, False

    dr, dc = MOVES[action]
    new_r = r + dr
    new_c = c + dc

    # hitting a wall: the agent stays where it is
    if new_r < 0 or new_r >= SIZE or new_c < 0 or new_c >= SIZE:
        new_r, new_c = r, c

    # the agent takes the gold when it walks on it
    if (new_r, new_c) == GOLD:
        has_gold = True

    new_state = (new_r, new_c, has_gold, has_arrow, wumpus_alive)

    if (new_r, new_c) in PITS or ((new_r, new_c) == WUMPUS and wumpus_alive):
        return new_state, -1000, True      # dead
    if (new_r, new_c) == START and has_gold:
        return new_state, 1000, True       # won
    return new_state, -1, False


start = (START[0], START[1], False, True, True)
print(step(start, "up"))
print(step(start, "left"))                       # wall
print(step((2, 3, False, True, True), "shoot up"))   # kills the Wumpus
print(step((2, 3, False, True, True), "up"))         # walks into the Wumpus

((2, 0, False, True, True), -1, False)
((3, 0, False, True, True), -1, False)
((2, 3, False, False, False), -10, False)
((1, 3, False, True, True), -1000, True)


## Q-table and epsilon-greedy

`Q[(state, action)]` = how good it is to do `action` in `state`. At the start everything is 0.

Epsilon-greedy: with probability `epsilon` the agent tries a random action (exploration),
otherwise it takes the best action it knows (exploitation).
At the beginning `epsilon` is 1 (only random actions), then it goes down, so the agent explores less and less (GLIE).

In [4]:
Q = {}
for r in range(SIZE):
    for c in range(SIZE):
        for has_gold in [False, True]:
            for has_arrow in [False, True]:
                for wumpus_alive in [False, True]:
                    for a in ACTIONS:
                        Q[((r, c, has_gold, has_arrow, wumpus_alive), a)] = 0


def best_action(state):
    best = ACTIONS[0]
    for a in ACTIONS:
        if Q[(state, a)] > Q[(state, best)]:
            best = a
    return best


def choose_action(state, epsilon):
    if random.random() < epsilon:
        return random.choice(ACTIONS)    # explore
    return best_action(state)            # exploit

## Q-learning

After every step:

Q(s, a) = Q(s, a) + ALPHA * (reward + GAMMA * max Q(s', a') - Q(s, a))

In [5]:
ALPHA = 0.1          # learning rate
GAMMA = 0.95         # discount factor
EPSILON_MIN = 0.01   # exploration at the end
EPISODES = 20000
MAX_STEPS = 100

rewards = []
wins = []
for episode in range(EPISODES):
    # epsilon goes down from 1 (all random) to EPSILON_MIN
    epsilon = max(EPSILON_MIN, 1 - episode / (EPISODES / 2))
    state = (START[0], START[1], False, True, True)
    total = 0
    for t in range(MAX_STEPS):
        action = choose_action(state, epsilon)
        new_state, reward, done = step(state, action)

        if done:
            target = reward
        else:
            target = reward + GAMMA * Q[(new_state, best_action(new_state))]
        Q[(state, action)] = Q[(state, action)] + ALPHA * (target - Q[(state, action)])

        total += reward
        state = new_state
        if done:
            break
    rewards.append(total)
    wins.append(reward == 1000)

# average reward and wins every 2000 games
for i in range(0, EPISODES, 2000):
    avg = sum(rewards[i:i + 2000]) / 2000
    won = sum(wins[i:i + 2000]) / 2000 * 100
    print(f"games {i + 1:>5}-{i + 2000:>5}: average reward = {avg:7.1f}   wins = {won:5.1f}%")

games     1- 2000: average reward = -1085.2   wins =   0.0%
games  2001- 4000: average reward = -1068.7   wins =   0.0%
games  4001- 6000: average reward = -1055.7   wins =   0.5%
games  6001- 8000: average reward =  -649.0   wins =  18.8%
games  8001-10000: average reward =   385.6   wins =  67.5%
games 10001-12000: average reward =   931.8   wins =  96.8%
games 12001-14000: average reward =   940.2   wins =  97.4%
games 14001-16000: average reward =   919.2   wins =  95.9%
games 16001-18000: average reward =   918.0   wins =  96.0%
games 18001-20000: average reward =   919.2   wins =  96.1%


## The learned policy

In [6]:
SYMBOLS = {"up": "^", "down": "v", "left": "<", "right": ">",
           "shoot up": "*^", "shoot down": "*v", "shoot left": "*<", "shoot right": "*>"}


def print_policy(has_gold, has_arrow, wumpus_alive):
    for r in range(SIZE):
        row = ""
        for c in range(SIZE):
            if (r, c) in PITS:
                row += " P "
            elif (r, c) == WUMPUS and wumpus_alive:
                row += " W "
            else:
                row += f"{SYMBOLS[best_action((r, c, has_gold, has_arrow, wumpus_alive))]:>2} "
        print(row)


print("Before shooting (arrow, Wumpus alive):")
print_policy(False, True, True)
print()
print("After killing the Wumpus, without the gold:")
print_policy(False, False, False)
print()
print("With the gold:")
print_policy(True, False, False)

Before shooting (arrow, Wumpus alive):
 <  <  P  ^ 
 v  P *>  W 
 >  >  ^  < 
 >  ^  P  ^ 

After killing the Wumpus, without the gold:
 >  ^  P  ^ 
 ^  P  >  ^ 
 ^  >  ^  ^ 
 >  v  P  ^ 

With the gold:
 v  >  P  v 
 v  P  v  v 
 v  <  <  < 
 ^  <  P  ^ 


## Play one game with the learned policy (no exploration)

In [7]:
state = (START[0], START[1], False, True, True)
total = 0
for t in range(MAX_STEPS):
    action = best_action(state)
    state, reward, done = step(state, action)
    total += reward
    print(f"{t + 1:>2} {action:<12} -> {state} {reward}")
    if done:
        break

print("total reward:", total)

 1 right        -> (3, 1, False, True, True) -1
 2 up           -> (2, 1, False, True, True) -1
 3 right        -> (2, 2, False, True, True) -1
 4 up           -> (1, 2, False, True, True) -1
 5 shoot right  -> (1, 2, False, False, False) -10
 6 right        -> (1, 3, False, False, False) -1
 7 up           -> (0, 3, True, False, False) -1
 8 down         -> (1, 3, True, False, False) -1
 9 down         -> (2, 3, True, False, False) -1
10 left         -> (2, 2, True, False, False) -1
11 left         -> (2, 1, True, False, False) -1
12 left         -> (2, 0, True, False, False) -1
13 down         -> (3, 0, True, False, False) 1000
total reward: 979
